In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
load_dotenv()

llm = ChatGoogleGenerativeAI(model='gemini-3.6-flash')

In [5]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [6]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [9]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': "I was going to tell you a joke about pizza… \n\nBut it’s a bit too **cheesy**, and honestly, I don't think I can **deliver** it right!",
   'extras': {'signature': 'ErQYCrEYARFNMg/iBJv2xCkQIw53AHkkVi5REL46+ZaNZtGfARsTXuuDB33PSSz6aZifDN/hA1fm7fryqHkuEpAJ5vu+Rw+LCdBGjYY15FBQMXuZl3Igg6+JHBvfTk9tRgye5GlBoI3A9bNgcJ7ouT49D3kk00kZkf82su6snwI1PDMJyhQxk4yzpSsedO9dU/MSccaNUzrATruvwfA1pgJUOdAKsXbX39pVsQb3cfSjDJsrvrllsKkm9Z5m8j6aMsZIx5vES1kqsRDqmjSGHzdlkj2Torvb9fUQ8NMmDFfjJrFoMJzU/jVR1g3aqPmNmhlbkS88IE7xYsaP2acdRKxrrxWvwp/SxuAk14pMs88GM8/S9R/TlTZ6IK1Wy/1tbwBrd41nMnkM5+7IyKd+g2+w0i/2ipmQ7uI/wxoJcrUUhAQpT4FlrtU/JCQv6A3xT2uz6oelYSA0XL0Fvu7ujJ0rsShzELxGXo6aWmw/cmoxRx+IiI3MhMUfQnSS56ctfRcSteoZTRa7rx7hlIY7oetzIQmduIkgpVzQPfg2wHq6KlnSs9HcyWZB78i/PgK4Xw7CHvbYJ+96fLwBf/6E0F8U36/AUtxhnljZU6aazFS43FzsxEaPvnYUDbyWHLeFsJ4jhnyobASAAPbBB8pV1C69A6wYplCKkAWQBly1zVLH93blnE5NZcmQQp9xldHE+K6+KoVBCtDd/KE5Z+K2wsbZYyNNjJtjLrKWNs2VQhToySyMAFZUSleLlJP8M9Ze9muoP1Wm3

In [10]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': "I was going to tell you a joke about pizza… \n\nBut it’s a bit too **cheesy**, and honestly, I don't think I can **deliver** it right!", 'extras': {'signature': 'ErQYCrEYARFNMg/iBJv2xCkQIw53AHkkVi5REL46+ZaNZtGfARsTXuuDB33PSSz6aZifDN/hA1fm7fryqHkuEpAJ5vu+Rw+LCdBGjYY15FBQMXuZl3Igg6+JHBvfTk9tRgye5GlBoI3A9bNgcJ7ouT49D3kk00kZkf82su6snwI1PDMJyhQxk4yzpSsedO9dU/MSccaNUzrATruvwfA1pgJUOdAKsXbX39pVsQb3cfSjDJsrvrllsKkm9Z5m8j6aMsZIx5vES1kqsRDqmjSGHzdlkj2Torvb9fUQ8NMmDFfjJrFoMJzU/jVR1g3aqPmNmhlbkS88IE7xYsaP2acdRKxrrxWvwp/SxuAk14pMs88GM8/S9R/TlTZ6IK1Wy/1tbwBrd41nMnkM5+7IyKd+g2+w0i/2ipmQ7uI/wxoJcrUUhAQpT4FlrtU/JCQv6A3xT2uz6oelYSA0XL0Fvu7ujJ0rsShzELxGXo6aWmw/cmoxRx+IiI3MhMUfQnSS56ctfRcSteoZTRa7rx7hlIY7oetzIQmduIkgpVzQPfg2wHq6KlnSs9HcyWZB78i/PgK4Xw7CHvbYJ+96fLwBf/6E0F8U36/AUtxhnljZU6aazFS43FzsxEaPvnYUDbyWHLeFsJ4jhnyobASAAPbBB8pV1C69A6wYplCKkAWQBly1zVLH93blnE5NZcmQQp9xldHE+K6+KoVBCtDd/KE5Z+K2wsbZYyNNjJtjLrKWNs2VQhToySyMAFZUSleLlJP

In [11]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': "I was going to tell you a joke about pizza… \n\nBut it’s a bit too **cheesy**, and honestly, I don't think I can **deliver** it right!", 'extras': {'signature': 'ErQYCrEYARFNMg/iBJv2xCkQIw53AHkkVi5REL46+ZaNZtGfARsTXuuDB33PSSz6aZifDN/hA1fm7fryqHkuEpAJ5vu+Rw+LCdBGjYY15FBQMXuZl3Igg6+JHBvfTk9tRgye5GlBoI3A9bNgcJ7ouT49D3kk00kZkf82su6snwI1PDMJyhQxk4yzpSsedO9dU/MSccaNUzrATruvwfA1pgJUOdAKsXbX39pVsQb3cfSjDJsrvrllsKkm9Z5m8j6aMsZIx5vES1kqsRDqmjSGHzdlkj2Torvb9fUQ8NMmDFfjJrFoMJzU/jVR1g3aqPmNmhlbkS88IE7xYsaP2acdRKxrrxWvwp/SxuAk14pMs88GM8/S9R/TlTZ6IK1Wy/1tbwBrd41nMnkM5+7IyKd+g2+w0i/2ipmQ7uI/wxoJcrUUhAQpT4FlrtU/JCQv6A3xT2uz6oelYSA0XL0Fvu7ujJ0rsShzELxGXo6aWmw/cmoxRx+IiI3MhMUfQnSS56ctfRcSteoZTRa7rx7hlIY7oetzIQmduIkgpVzQPfg2wHq6KlnSs9HcyWZB78i/PgK4Xw7CHvbYJ+96fLwBf/6E0F8U36/AUtxhnljZU6aazFS43FzsxEaPvnYUDbyWHLeFsJ4jhnyobASAAPbBB8pV1C69A6wYplCKkAWQBly1zVLH93blnE5NZcmQQp9xldHE+K6+KoVBCtDd/KE5Z+K2wsbZYyNNjJtjLrKWNs2VQhToySyMAFZUSleLlJ

In [12]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': [{'type': 'text',
   'text': 'What do you call a fake noodle?\n\nAn **im-pasta**! 🍝',
   'extras': {'signature': 'EvEdCu4dARFNMg/LZ8AJHh2A7eUQFAsdUcklNK2sH2icZFIYeG6M9eeCfb+y5aP2L7tL9s5pVomXQjz1qWlKpefZJ7PBcGlgujihXWBPctNHOuJHnXH2GVcr+JEIxOCPpzFuFqhj5dqWbWKjJx42kgkx22zrlFQKUn+RJVsBfUOB4BoDiYMCvV0QTxxckbdSAMXb03t12IDStJt4uNpqQlhMwdoluxGfvdpqfc9OxOLkrxKsPrxbPE8ufDpTe0pmb292s/7zmfVCiev9+FfKUeM4RpAaHJBVLRTfiOxYo6YnmCeMY9CiLxKvGAdKK6hb2XU3aF4WSXrutqFuPujShB+FHGeIIaP4rrmHfnNIfrkhSDYkoF/NZOvI3DWJXYvteVS+TM67s2FO4u8jD2zHz1TBt5ou0NNMrUV51T0qjH+7VglhFGLU89svn86QUA4Q1bC1kfn+ykx6rjTHdX7rGK1uSmO+L48iP9fm9LpanehiirC+p6Yn7Gi+JilAkUF9TZ5XPjX9P9/3qovolTMCM/+dm1Tes/NKdSXFz9E969NTAg19yAtJzu58o9CeCmNiVV9JIPQ42cUP8feKhhqkfzxPApTGHFJuYMFcxlSUTN/G4MRLGe0lrD5FVLi5rvGKFIwLEQyj3skUVIQ8tW1jUT878zsxF4dXwutmAxN/nmQuKWAuikJ5BFfAUtK/ogZID9NHxrOj9Bp3Kz2KwZpm7J/LIZJIowDLEqXpjYaO6ZuCn1au67WcDOt2vPpm9IQcZLDTtxLzIPbqbJLzu5rmwOJ5v2QYl+5KE9sdFpcI8MRXHUgyC9TBeiePIMggIiHWvZuXVMhYQ//L5N74JdGTtkNfVW

In [13]:

workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': "I was going to tell you a joke about pizza… \n\nBut it’s a bit too **cheesy**, and honestly, I don't think I can **deliver** it right!", 'extras': {'signature': 'ErQYCrEYARFNMg/iBJv2xCkQIw53AHkkVi5REL46+ZaNZtGfARsTXuuDB33PSSz6aZifDN/hA1fm7fryqHkuEpAJ5vu+Rw+LCdBGjYY15FBQMXuZl3Igg6+JHBvfTk9tRgye5GlBoI3A9bNgcJ7ouT49D3kk00kZkf82su6snwI1PDMJyhQxk4yzpSsedO9dU/MSccaNUzrATruvwfA1pgJUOdAKsXbX39pVsQb3cfSjDJsrvrllsKkm9Z5m8j6aMsZIx5vES1kqsRDqmjSGHzdlkj2Torvb9fUQ8NMmDFfjJrFoMJzU/jVR1g3aqPmNmhlbkS88IE7xYsaP2acdRKxrrxWvwp/SxuAk14pMs88GM8/S9R/TlTZ6IK1Wy/1tbwBrd41nMnkM5+7IyKd+g2+w0i/2ipmQ7uI/wxoJcrUUhAQpT4FlrtU/JCQv6A3xT2uz6oelYSA0XL0Fvu7ujJ0rsShzELxGXo6aWmw/cmoxRx+IiI3MhMUfQnSS56ctfRcSteoZTRa7rx7hlIY7oetzIQmduIkgpVzQPfg2wHq6KlnSs9HcyWZB78i/PgK4Xw7CHvbYJ+96fLwBf/6E0F8U36/AUtxhnljZU6aazFS43FzsxEaPvnYUDbyWHLeFsJ4jhnyobASAAPbBB8pV1C69A6wYplCKkAWQBly1zVLH93blnE5NZcmQQp9xldHE+K6+KoVBCtDd/KE5Z+K2wsbZYyNNjJtjLrKWNs2VQhToySyMAFZUSleLlJP

In [14]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': "I was going to tell you a joke about pizza… \n\nBut it’s a bit too **cheesy**, and honestly, I don't think I can **deliver** it right!", 'extras': {'signature': 'ErQYCrEYARFNMg/iBJv2xCkQIw53AHkkVi5REL46+ZaNZtGfARsTXuuDB33PSSz6aZifDN/hA1fm7fryqHkuEpAJ5vu+Rw+LCdBGjYY15FBQMXuZl3Igg6+JHBvfTk9tRgye5GlBoI3A9bNgcJ7ouT49D3kk00kZkf82su6snwI1PDMJyhQxk4yzpSsedO9dU/MSccaNUzrATruvwfA1pgJUOdAKsXbX39pVsQb3cfSjDJsrvrllsKkm9Z5m8j6aMsZIx5vES1kqsRDqmjSGHzdlkj2Torvb9fUQ8NMmDFfjJrFoMJzU/jVR1g3aqPmNmhlbkS88IE7xYsaP2acdRKxrrxWvwp/SxuAk14pMs88GM8/S9R/TlTZ6IK1Wy/1tbwBrd41nMnkM5+7IyKd+g2+w0i/2ipmQ7uI/wxoJcrUUhAQpT4FlrtU/JCQv6A3xT2uz6oelYSA0XL0Fvu7ujJ0rsShzELxGXo6aWmw/cmoxRx+IiI3MhMUfQnSS56ctfRcSteoZTRa7rx7hlIY7oetzIQmduIkgpVzQPfg2wHq6KlnSs9HcyWZB78i/PgK4Xw7CHvbYJ+96fLwBf/6E0F8U36/AUtxhnljZU6aazFS43FzsxEaPvnYUDbyWHLeFsJ4jhnyobASAAPbBB8pV1C69A6wYplCKkAWQBly1zVLH93blnE5NZcmQQp9xldHE+K6+KoVBCtDd/KE5Z+K2wsbZYyNNjJtjLrKWNs2VQhToySyMAFZUSleLlJ

Time Travel

In [15]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f199719-f968-6931-8000-7892c813e571"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f199719-f968-6931-8000-7892c813e571'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f199719-f968-6931-8000-7892c813e571"}})

In [ ]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the pizza go to therapy?\n\nBecause it had serious **crust** issues! 🍕', 'extras': {'signature': 'EvYaCvMaARFNMg9IR0X2HIzQcw/dRN++RJg0GOaHEu9tRY8FrTHrX0qeb3Kn0UDe+0dcm7m9CU45mtjBGCFFKWuWSzQB1ilWPCaWhlrd1XpfWt6wXIj5WBqda2qjqtA486bReUMmUPPlBA3onJ2VUyiFyEKB6ymT3DRi+b8Z8TZDNVUGHZccjIGBDiGo5hd6S2Q0gi0Jhw6ugfD6IvFuc0qIFCl/S57GRdUSgx0t2NnWIyskI+6JLWjtcC/tPK/fQ7WNUjDcI0Ctji/RyT1gW/dfT6p/SgpVtwJEpm/c0KBDs0kwXiMcSV9mZtniULnhoshUybf7DtRKw4N1YRE6HtaM4SNbIXSZa/pktwX5nJOjD2ZWylzibYY1mU3ZWs9zSrZBKEP2qE71VK6Gx5bklpa8szXEWqGLwnmog/8GrKliahI2MGRH3nvvjQCavpYnbjjWOINA5dTkJdbjJadRXfU9rCzuAgDF12d92uTuJ6lGznM5TYdMauF14Hvxbrl37LdGF3pI768/wOI22MH540c2ms/d5aEaZhLkCwkfSheUifxdVcmvs4NFtjJuJCLqh8Wo8kyqRzumKLcqpxwF2SPbieA+G6IhBw+6ZMinvw8eujJ46hDEilboxV/w9Zqy27dkP/irjoTsrePrHfMTGWylXJXE1z73RoZSPZIbYluy660J4I7NH6AOOEdVz9kjl6AyUPH79oIeZeROTTvcCgDAkh445QJLXVYaR2gLgzA11obxC3ycvv1wwNImgPv+PN04djzmIWLNdquCaahuMhTN5QcDYbz+KeAWzXM2ZhB4bfg3f/

Updating State

In [18]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f199719-f962-6b5e-bfff-7f1d027d028b", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f199736-2bc3-66e4-8000-3879f63a67cf'}}

In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199736-2bc3-66e4-8000-3879f63a67cf'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-08-16T13:07:08.198671+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199719-f962-6b5e-bfff-7f1d027d028b'}}, tasks=(PregelTask(id='daeb8408-f97b-c7cb-5129-ce64b2952c2b', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': "I was going to tell you a joke about pizza… \n\nBut it’s a bit too **cheesy**, and honestly, I don't think I can **deliver** it right!", 'extras': {'signature': 'ErQYCrEYARFNMg/iBJv2xCkQIw53AHkkVi5REL46+ZaNZtGfARsTXuuDB33PSSz6aZifDN/hA1fm7fryqHkuEpAJ5vu+Rw+LCdBGjYY15FBQMXuZl3Igg6+JHBvfTk9tRgye5GlBoI3A9b

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f199719-f962-6b5e-bfff-7f1d027d028b"}})

In [ ]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': "I was going to tell you a joke about pizza... \n\n...but it’s a little too **cheesy**, and I don't think I can **deliver** it right!", 'extras': {'signature': 'Et0VCtoVARFNMg9St39sClAdKVVrht2377Bc03RVQFRR7CIhQBz5VFbeSS53UwSnsbfzzSomYyxpfxP9F8IedQMfbiLtHFHPoMcriTUdXhJi3z//W6NQf0glS+llFNxB1/mlT8fpy9HgX/PRANSmp5DBUD/k/o8FBIUYoikwOp916Tr5P9+yTux39gJ06YjTZ5qBnNBnOwsGMyppyYtDlZKIXRAy/5mO3GDCRdQ42pzYkISaXQALUv+8bmSVrrmFav+gaoOCdlaSVU1bC7gTj9M33P9u/RDfVZmt+aj4o6tVPCVhKJWzpQ00WXo/ub6nKK8TGHZO+nJ3D3IswWxBZk10R1oJdbQsAM0bouUdTIiqwH2B0nY+LfEHclu+7G5Oeqyg7tPKhJCsNJBTg03qBmTk0idn7xQz3nToKMm6rYIVBPsLhvaCAYTrwkX0Vwo2uSpYybRmempJhQ+fhMnDKv17Lzv5K/C1OJcd97LHHBspHIuvSzIXDKFyZ7ka/3/3t3gddC3/eKeIlP+fHjqRPOYnqte7IpPWctdYN5kNUsq9Xg5AUS8h7KpRejV47ZhVGINgIVym9muxWZkvRxC3G2zUHZupW6Gsbp3Sk4wLPP/MKH/iWhXOe+kv2J3F/ynyCKHL8c0jk4mKmZ1UBAGG4DeHMm5/wAIDfiNq3ZY3MNe//DR77ujNE6eDeMi+eJLOzcEE0yWTmEpCuDwQn3ypZ3JcqsMOrnIYI/hQyloc7CK86waIT7jrHhGzgKhd

Fault Tolerance

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [ ]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [ ]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [ ]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:

try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...


NameError: name 'graph' is not defined

In [ ]:

# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...


NameError: name 'graph' is not defined

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))